In [2]:
from dotenv import load_dotenv
load_dotenv()
from typing import Annotated, TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, START,END, add_messages
from langchain.chat_models import BaseChatModel, init_chat_model
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from typing import Literal
import langchain_openai

/Users/kaushik/Documents/OTHER_PROJECTS/GenAI-LLLM/research-asst/venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
# SYSTEM DESIGN FOR MULIT-AGENT RESEARH ASSISTANT
# User Request -> router -> retriever agent(use tools, RAG, vector embeddings) (or ->direct answer node)-> summarizer, critic, synthesizer

In [ ]:
class ResearchState(TypedDict): 
    query : str
    retrieved_docs : List[str]
    sources        : List[str]  
    summary : str
    critique : Dict[str, Any]
    final_answer : str
    iteration: int


research_llm = init_chat_model(
    model = 'gpt-4.1-nano',
    temperature = 0.1
)


### Router Node

In [5]:

def router_node(state: ResearchState, research_llm: BaseChatModel) -> Literal["retriever", "direct_answer"]:
    """ 
    Decides whether the query requires retrieval from external knowledge base

    Return:
    "retriever" : query needs external documents 
    "direct_answer" : query can be satisfied with common knowledge
    """

    prompt = f"""
        You are a query router for a research assistant agent.

        You decide if the given query needs document retrieval or can be answered from general knowledge

        Query : {state["query"]}

        Query -> if asks about specific facts, events, details, niche topic, scientific nitpicks : "RETRIEVE"
        Query -> if asks general information that is conceptual, definitional, or widely known : "DIRECT"

        Respond with exactly one word: RETRIEVE or DIRECT
    """
    decision = research_llm.invoke(prompt)

    return "retriever" if decision.content.strip() == "RETRIEVE" else "direct_answer"


### Retriever Node

In [6]:
def retriever_node(state : ResearchState, vector_db) -> dict:
    """ Fetched the top-k relevant documents pertaining the query.
    
    Reads : state["query"]
    writes : state["retreived_docs"]
    """

    query = state["query"]
    docs = vector_db.similarity_search(query, k = 5)

    clean_docs = [doc.page_content for doc in docs]

    return {
        "retrieved_docs" : clean_docs
    }

### Summarizer Node

In [7]:
def summarizer_node(state: ResearchState, llm: BaseChatModel) -> dict:
    """ Condenses retreived documents into structured key points
    
    Reads : state["retrieved_docs"]
    Write:  state["summary]
    
    """

    docs = state["retrieved_docs"]
    context = "\n\n---\n\n".join(docs)

    prompt = f"""
        You are a research summarizer

        Summarixe the following source documents clearly, into structured key points that directly address the user's query.

        Source documents:
        {context}

        Instructions:
        1) Extract only most relevant information
        2) Remove redundancy across documents
        3) Organize by topic or importance
        4) Be concise but complete
        """
    
    summary = research_llm.invoke(prompt).content

    return{
        "summary" : summary
    }

### Critic Node

In [8]:
from pydantic import BaseModel, Field

class CritiqueOutput(BaseModel):
    score: float = Field(ge=0.0, le=1.0)
    issues: List[str]
    needs_revision: bool
 

def critic_node(state: ResearchState, research_llm: BaseChatModel) -> dict:
    score: float = Field(ge=0.0, le=1.0, description = "Qualit score from 0 to 1")
    issues: List[str] = Field(description = "List of specific problems with the summary")
    needs_revision: bool = Field(descriptio = "True if summary should be revised, else False")
    """ 
    Evaluate the summary and return a structured critic

    Reads:  state["query"], state["summary"], state["iteration"]
    Writes: state["critique"], state["iteration"]
    
    """
    query = state["query"]
    summary = state["summary"]
    prompt = f"""
        You are a ciritical evaluator for a research assistant.
        Evaluate the following summary against the user's original query
        Query: {query}
 
    Summary : {summary}

Assess the summary on :
1) Relevance : does it answer the query?
2) Completeness: are key points missing?
3) Accuracy: are any claims unsupported or vague?
4) Clarity: is it well-structured and easy to read?

Be specific, List concrete issues, not generic complaints

 """
    
    structured_llm = research_llm.with_structured_output(CritiqueOutput)
    critic : critic_node = structured_llm.invoke(prompt)

    return {
        "critique" : critic.model_dump(),
        "iteration" : state.get("iteration", 0)+1
    }


MAX_ITERATIONS = 3
 
def should_refine(state: ResearchState) -> Literal["refine", "synthesizer"]:
    """
    Decides whether to loop back through refine_node or proceed to synthesizer.
 
    Returns:
        "refine"      → critic flagged issues and we haven't hit the iteration cap
        "synthesizer" → quality is good enough, or we've run out of attempts
    """
    critique = state["critique"]
    iteration = state["iteration"]
 
    if critique["needs_revision"] and iteration < MAX_ITERATIONS:
        return "refine"
 
    return "synthesizer"
 

### Refine Node

In [9]:
def refine_node(state: ResearchState, llm: BaseChatModel) -> dict:
    """
    Rewrites the summary using the critic's specific feedback.
 
    Reads:  state["summary"], state["critique"]
    Writes: state["summary"]  (overwrites with improved version)
    """
    summary = state["summary"]
    issues = state["critique"]["issues"]
    score = state["critique"]["score"]
 
    # Format issues as a numbered list for the prompt
    formatted_issues = "\n".join(f"{i+1}. {issue}" for i, issue in enumerate(issues))
 
    prompt = f"""
    You are a research editor improving a summary based on reviewer feedback.
 
    Current quality score: {score:.2f} / 1.0
 
    Issues identified by the critic:
    {formatted_issues}
 
    Original summary:
    {summary}
 
    Instructions:
    - Fix each issue listed above
    - Do not remove content that was already correct
    - Keep the structure clean and well-organised
    - Do not add unsupported claims
    """
 
    improved_summary = llm.invoke(prompt).content
 
    return {
        "summary": improved_summary
    }

### Synthesizer Node

In [10]:
def synthesizer_node(state: ResearchState, llm: BaseChatModel) -> dict:
    """
    Produces the final structured answer from the polished summary.
 
    Reads:  state["query"], state["summary"]
    Writes: state["final_answer"]
    """
    query = state["query"]
    summary = state["summary"]
 
    prompt = f"""
    You are a research assistant writing a final answer for a user.
 
    User's query: {query}
 
    Research summary:
    {summary}
 
    Instructions:
    - Answer the query directly and completely
    - Use the summary as your source of truth
    - Use clear headings if the answer has multiple parts
    - Cite the key points from the summary where relevant
    - Be concise — no padding or repetition
    """
 
    final_answer = llm.invoke(prompt).content
 
    return {
        "final_answer": final_answer
    }

### Direct Answer Node

In [11]:
def direct_answer_node(state: ResearchState, llm: BaseChatModel) -> dict:
    """
    Answers the query directly from LLM knowledge, skipping retrieval.
    Writes to state["summary"] so synthesizer_node handles final formatting.
 
    Reads:  state["query"]
    Writes: state["summary"]
    """
    query = state["query"]
 
    prompt = f"""
    You are a knowledgeable research assistant.
 
    Answer the following query from your general knowledge.
    Be accurate, structured, and thorough.
 
    Query: {query}
    """
 
    # Write to summary so synthesizer_node can handle final formatting
    # consistently regardless of which path was taken
    answer = llm.invoke(prompt).content
 
    return {
        "summary": answer
    }

In [12]:
from functools import partial

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_db = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

builder = StateGraph(ResearchState)

builder.add_node("retriever",     partial(retriever_node,     vector_db=vector_db))
builder.add_node("direct_answer", partial(direct_answer_node,  llm=research_llm))
builder.add_node("summarizer",    partial(summarizer_node,    llm=research_llm))
builder.add_node("critic",        partial(critic_node,        research_llm=research_llm))
builder.add_node("refine",        partial(refine_node,        llm=research_llm))
builder.add_node("synthesizer",   partial(synthesizer_node,   llm=research_llm))

builder.add_conditional_edges(START, partial(router_node, research_llm=research_llm))

builder.add_edge("retriever", "summarizer")
builder.add_edge("summarizer", "critic")
builder.add_conditional_edges("critic", should_refine)
builder.add_edge("refine", "critic")
builder.add_edge("direct_answer", "synthesizer")
builder.add_edge("synthesizer", END)

graph = builder.compile()


In [13]:
if __name__ == "__main__":
    result = graph.invoke({"query": "what are the resolution levels a tumor cell microscopy is captured with?"})
    print(result["final_answer"])


/var/folders/w_/6x67ns8x41q1_vrmr37yf27r0000gn/T/ipykernel_42504/2387384729.py:12: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descriptio'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  needs_revision: bool = Field(descriptio = "True if summary should be revised, else False")


The resolution levels at which tumor cell microscopy images are captured typically involve a spatial resolution of approximately 0.5 micrometers per pixel. This resolution ensures sufficient detail for cellular analysis, allowing for accurate segmentation of nuclei, cytoplasm, and plasma membranes (as noted in the imaging resolution and tiling strategy). The images are divided into tiles of about 64 μm, roughly a quarter of a cell’s diameter, to encode cellular features effectively. This tiling approach preserves spatial resolution relevant to cellular phenotypes, enabling detailed phenotypic and heterogeneity analyses.


In [14]:
result


{'query': 'what are the resolution levels a tumor cell microscopy is captured with?',
 'retrieved_docs': ['918 \ndetermined to be of insufficient quality. Raw microscopy tiles (in .rcpnl format) were stitched, registered, \n919 \nsegmented, and quantified using the MCMICRO image processing pipeline28. Cells in the image were \n920 \nquantified according to three different cell segmentation mask objects: nucleus, cytoplasm, and plasma \n921 \nmembrane. To ensure the most accurate segmentation-based signal quantification, the choice of mask selection \n922',
  '504 \nbit depth. Single cells in this image were approximated using 11x11µm image patches. Despite the lower bit \n505 \ndepth and relatively small number of cells in this image (~10K), the Leiden algorithm identified 19 VAE image \n506 \npatch clusters (V110-V1118) that were also well separated in UMAP space (Extended Data Fig. 6e). Multiple \n507 \ntumor cell phenotypes were identified in this sample based on differences in mark

In [15]:
result["final_answer"]

'The resolution levels at which tumor cell microscopy images are captured typically involve a spatial resolution of approximately 0.5 micrometers per pixel. This resolution ensures sufficient detail for cellular analysis, allowing for accurate segmentation of nuclei, cytoplasm, and plasma membranes (as noted in the imaging resolution and tiling strategy). The images are divided into tiles of about 64 μm, roughly a quarter of a cell’s diameter, to encode cellular features effectively. This tiling approach preserves spatial resolution relevant to cellular phenotypes, enabling detailed phenotypic and heterogeneity analyses.'

In [22]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__(<p>__start__</p>)
	retriever(retriever)
	direct_answer(direct_answer)
	summarizer(summarizer)
	critic(critic)
	refine(refine)
	synthesizer(synthesizer)
	__end__(<p>__end__</p>)
	__start__ -.-> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

